# Carry-On Shopping Answer Analysis  
## Comparison 01, baseline-based version  
### 2023/2024 Original Baseline vs One-Product 2026 Current Replacement

This notebook analyzes:

**2023/2024 original baseline vs 2026 current one-product replacement**

Purpose: check how much the actual product page changed over time.  
Interpretation: general website/text change.

## Key design

You now have a direct baseline file:

```text
code/data/results/shopping_answers/baseline/baseline.txt
```

The baseline file should contain the shopping-answer output generated from the five older original product pages:

```text
Monos 2023 original
INUSA 2023 original
BEIS 2024 original
Delsey 2024 original
Travelpro 2024 original
```

Each file in Comparison 01 is treated as a one-product-current counterfactual run:

```text
beis_2023vs2026.txt
```

means:

- BEIS uses the 2026 current product-page text.
- The other four products use the older 2023/2024 original product-page texts.

The product-level effect is computed as:

```text
effect for product p = metric for p in focal 2026 file - metric for p in all-original baseline
```

Positive deltas mean that the product became more visible, more cited, or more prominent when its own page was replaced by the 2026 current version.


## Expected folder structure

```text
code/
└── data/
    └── results/
        └── shopping_answers/
            ├── baseline/
            │   └── baseline.txt
            └── 01_2023_2024_original_vs_2026_current/
                ├── beis_2023vs2026.txt
                ├── delsey_2023vs2026.txt
                ├── inusa_2023vs2026.txt
                ├── monos_2023vs2026.txt
                └── travel_2023vs2026.txt
```

Optional product-page text folders used for PAIS-style text similarity:

```text
code/data/filtered/old/
code/data/filtered/current/
```


## CSV outputs

The notebook saves CSV files to:

```text
code/data/results/tables/01_2023_2024_original_vs_2026_current_baseline_based/
```

Saved files:

```text
baseline_parsed_answers.csv
current_parsed_answers_by_file.csv
baseline_question_product_metrics.csv
current_question_product_metrics.csv
baseline_product_summary.csv
current_product_summary.csv
current_vs_baseline_delta_by_product.csv
overall_current_vs_baseline_summary.csv
product_text_load_status.csv
```


In [13]:
from pathlib import Path
import re
import json
import math
import numpy as np
import pandas as pd

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 200)


In [14]:
# =============================
# Configuration
# =============================

PRODUCTS = ["Monos", "BEIS", "Travelpro", "Delsey", "INUSA"]
N_PRODUCTS = len(PRODUCTS)

COMPARISON_NAME = "01_2023_2024_original_vs_2026_current"
OUTPUT_NAME = "01_2023_2024_original_vs_2026_current_baseline_based"

# If automatic path detection fails, manually set these:
# BASELINE_FILE = Path("data/results/shopping_answers/baseline/baseline.txt")
# COMPARISON_DIR = Path("data/results/shopping_answers/01_2023_2024_original_vs_2026_current")
BASELINE_FILE = None
COMPARISON_DIR = None

# Optional manual overrides for product-page text folders.
# These are used only for PAIS-style product-text-to-answer similarity.
OLD_TEXT_DIR = None
CURRENT_TEXT_DIR = None


In [15]:
# =============================
# Robust path detection
# =============================

def get_candidate_roots():
    cwd = Path.cwd().resolve()
    roots = []
    for p in [
        cwd,
        cwd.parent,
        cwd.parent.parent,
        cwd / "code",
        cwd.parent / "code",
        cwd.parent.parent / "code",
    ]:
        if p.exists() and p not in roots:
            roots.append(p)
    return roots

def find_dir(rel_candidates):
    for root in get_candidate_roots():
        for rel in rel_candidates:
            candidate = root / rel
            if candidate.exists() and candidate.is_dir():
                return candidate.resolve()
    return None

def find_file(rel_candidates):
    for root in get_candidate_roots():
        for rel in rel_candidates:
            candidate = root / rel
            if candidate.exists() and candidate.is_file():
                return candidate.resolve()
    return None

if BASELINE_FILE is None:
    BASELINE_FILE = find_file([
        Path("data/results/shopping_answers/baseline/baseline.txt"),
        Path("code/data/results/shopping_answers/baseline/baseline.txt"),
        Path("results/shopping_answers/baseline/baseline.txt"),
        Path("data/results/shopping_answers/00_baseline/baseline.txt"),
        Path("code/data/results/shopping_answers/00_baseline/baseline.txt"),
        Path("results/shopping_answers/00_baseline/baseline.txt"),
        Path("data/results/shopping_answers/00_all_original_2023_2024/baseline.txt"),
        Path("code/data/results/shopping_answers/00_all_original_2023_2024/baseline.txt"),
        Path("results/shopping_answers/00_all_original_2023_2024/baseline.txt"),
        Path("data/results/shopping_answers/00_all_original_2023_2024/carry_on_shopping_answers.txt"),
        Path("code/data/results/shopping_answers/00_all_original_2023_2024/carry_on_shopping_answers.txt"),
        Path("results/shopping_answers/00_all_original_2023_2024/carry_on_shopping_answers.txt"),
    ])
else:
    BASELINE_FILE = Path(BASELINE_FILE).resolve()

if COMPARISON_DIR is None:
    COMPARISON_DIR = find_dir([
        Path("data/results/shopping_answers") / COMPARISON_NAME,
        Path("code/data/results/shopping_answers") / COMPARISON_NAME,
        Path("results/shopping_answers") / COMPARISON_NAME,
    ])
else:
    COMPARISON_DIR = Path(COMPARISON_DIR).resolve()

if OLD_TEXT_DIR is None:
    OLD_TEXT_DIR = find_dir([
        Path("data/filtered/old"),
        Path("code/data/filtered/old"),
    ])
else:
    OLD_TEXT_DIR = Path(OLD_TEXT_DIR).resolve()

if CURRENT_TEXT_DIR is None:
    CURRENT_TEXT_DIR = find_dir([
        Path("data/filtered/current"),
        Path("code/data/filtered/current"),
    ])
else:
    CURRENT_TEXT_DIR = Path(CURRENT_TEXT_DIR).resolve()

print("Current working directory:", Path.cwd().resolve())
print("Detected baseline file:", BASELINE_FILE)
print("Detected comparison directory:", COMPARISON_DIR)
print("Detected old text directory:", OLD_TEXT_DIR)
print("Detected current text directory:", CURRENT_TEXT_DIR)

if BASELINE_FILE is None:
    raise FileNotFoundError(
        "Could not find baseline.txt. "
        "Please set BASELINE_FILE manually in the configuration cell."
    )

if COMPARISON_DIR is None:
    raise FileNotFoundError(
        "Could not find the comparison folder. "
        "Please set COMPARISON_DIR manually in the configuration cell."
    )

TXT_FILES = sorted(COMPARISON_DIR.glob("*.txt"))

print(f"Found {len(TXT_FILES)} one-product-current txt files:")
for p in TXT_FILES:
    print("-", p.name)

if not TXT_FILES:
    raise FileNotFoundError(f"No .txt files found in {COMPARISON_DIR}")


Current working directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\code
Detected baseline file: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline.txt
Detected comparison directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current
Detected old text directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\old
Detected current text directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\current
Found 5 one-product-current txt files:
- beis_2023vs2026.txt
- delsey_2023vs2026.txt
- inusa_2023vs2026.txt
- monos_2023vs2026.txt
- travel_2023vs2026.txt


In [16]:
# =============================
# Output directory
# =============================

# Based on your structure, if COMPARISON_DIR is:
# code/data/results/shopping_answers/01_...
# then tables go to:
# code/data/results/tables/01_..._baseline_based
RESULTS_DIR = COMPARISON_DIR.parents[1]  # .../results
OUTPUT_DIR = RESULTS_DIR / "tables" / OUTPUT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Results directory:", RESULTS_DIR)
print("CSV output directory:", OUTPUT_DIR)


Results directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results
CSV output directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\tables\01_2023_2024_original_vs_2026_current_baseline_based


## Parse answer files

The parser expects each answer file to contain five question blocks:

```text
Question 1:
Answer:
...

Question 2:
Answer:
...

Overall shopper takeaway:
...
```


In [17]:
# =============================
# Parsing functions
# =============================

QUESTION_BLOCK_RE = re.compile(
    r"Question\s*(\d+)\s*:\s*(.*?)(?=\n\s*Question\s*\d+\s*:|\n\s*Overall shopper takeaway\s*:|\Z)",
    flags=re.IGNORECASE | re.DOTALL,
)

OVERALL_RE = re.compile(
    r"Overall shopper takeaway\s*:\s*(.*)\Z",
    flags=re.IGNORECASE | re.DOTALL,
)

def clean_answer_block(block: str) -> str:
    block = block.strip()
    block = re.sub(r"^\s*Answer\s*:\s*", "", block, flags=re.IGNORECASE).strip()
    block = re.sub(r"\n{3,}", "\n\n", block)
    return block

def infer_focal_current_product_from_filename(filename: str) -> str:
    name = filename.lower()
    if "beis" in name or "béis" in name:
        return "BEIS"
    if "delsey" in name:
        return "Delsey"
    if "inusa" in name:
        return "INUSA"
    if "monos" in name:
        return "Monos"
    if "travel" in name or "travelpro" in name:
        return "Travelpro"
    return "Unknown"

def parse_answer_file(path: Path, run_type: str, focal_current_product: str = None) -> pd.DataFrame:
    text = path.read_text(encoding="utf-8", errors="replace")
    rows = []
    for m in QUESTION_BLOCK_RE.finditer(text):
        qid = int(m.group(1))
        answer = clean_answer_block(m.group(2))
        rows.append({
            "source_file": path.name,
            "run_type": run_type,
            "focal_current_product": focal_current_product,
            "comparison": COMPARISON_NAME,
            "question_id": qid,
            "answer_text": answer,
        })

    overall = ""
    om = OVERALL_RE.search(text)
    if om:
        overall = om.group(1).strip()

    df = pd.DataFrame(rows)
    if not df.empty:
        df["overall_shopper_takeaway"] = overall
    return df

baseline_answers = parse_answer_file(BASELINE_FILE, run_type="baseline_all_original", focal_current_product=None)

current_answers = pd.concat(
    [
        parse_answer_file(
            p,
            run_type="one_product_current",
            focal_current_product=infer_focal_current_product_from_filename(p.name),
        )
        for p in TXT_FILES
    ],
    ignore_index=True,
)

print("Baseline parsed rows:", len(baseline_answers))
display(baseline_answers)

print("Current counterfactual parsed rows:", len(current_answers))
display(current_answers.head(10))

print("Focal current products detected by file:")
display(current_answers[["source_file", "focal_current_product"]].drop_duplicates().sort_values("source_file"))


Baseline parsed rows: 5


,source_file,run_type,focal_current_product,comparison,question_id,answer_text,overall_shopper_takeaway
0,baseline.txt,baseline_all_original,None,01_2023_2024_original_vs_2026_current,1,"For a 3-5 day trip, I would recommend the Delsey CHATELET AIR 2.0 Carry-On Plus Spinner first because the page specifically says the Carry-On Plus is ideal for 3-5 day trips, f...","Choose Delsey for the most clearly stated 3-5 day trip fit, Travelpro for the strongest combination of durability, maneuverability, warranty, trial, and domestic-airline sizing..."
1,baseline.txt,baseline_all_original,None,01_2023_2024_original_vs_2026_current,2,"The Travelpro seems best if durability and protecting belongings matter most because its page describes an ultra-strong, easy-to-clean 100% polycarbonate shell that flexes on i...","Choose Delsey for the most clearly stated 3-5 day trip fit, Travelpro for the strongest combination of durability, maneuverability, warranty, trial, and domestic-airline sizing..."
2,baseline.txt,baseline_all_original,None,01_2023_2024_original_vs_2026_current,3,"The Travelpro appears easiest to move through airports, streets, and crowded spaces because its PrecisionGlide system is described as delivering precise control and effortless ...","Choose Delsey for the most clearly stated 3-5 day trip fit, Travelpro for the strongest combination of durability, maneuverability, warranty, trial, and domestic-airline sizing..."
3,baseline.txt,baseline_all_original,None,01_2023_2024_original_vs_2026_current,4,"BEIS seems best for packing organization because it includes a butterfly opening, a U-zip flap with one zip pocket and one frosted PVC zip pocket, a detachable compression flap...","Choose Delsey for the most clearly stated 3-5 day trip fit, Travelpro for the strongest combination of durability, maneuverability, warranty, trial, and domestic-airline sizing..."
4,baseline.txt,baseline_all_original,None,01_2023_2024_original_vs_2026_current,5,"For the best overall purchase confidence, I would choose Travelpro because it gives exact overall dimensions, a 45 L volume, sizer-bin testing for overhead bins on most major d...","Choose Delsey for the most clearly stated 3-5 day trip fit, Travelpro for the strongest combination of durability, maneuverability, warranty, trial, and domestic-airline sizing..."


Current counterfactual parsed rows: 25


,source_file,run_type,focal_current_product,comparison,question_id,answer_text,overall_shopper_takeaway
0,beis_2023vs2026.txt,one_product_current,BEIS,01_2023_2024_original_vs_2026_current,1,"For a 3-5 day trip, I would recommend the Delsey CHATELET AIR 2.0 Carry-On Plus Spinner as the most directly matched option because its page specifically says the Carry-On Plus...","For a straightforward 3-5 day carry-on, Delsey is the best direct match because the page explicitly names that trip length and includes useful organization and security feature..."
1,beis_2023vs2026.txt,one_product_current,BEIS,01_2023_2024_original_vs_2026_current,2,"If durability and protecting belongings are the top priorities, I would recommend Travelpro first, with Delsey as a close second. Travelpro uses an ultra-strong, easy-to-clean ...","For a straightforward 3-5 day carry-on, Delsey is the best direct match because the page explicitly names that trip length and includes useful organization and security feature..."
2,beis_2023vs2026.txt,one_product_current,BEIS,01_2023_2024_original_vs_2026_current,3,Travelpro seems easiest to move through airports and crowded spaces because its page gives the most specific mobility system: PrecisionGlide® for precise control and effortless...,"For a straightforward 3-5 day carry-on, Delsey is the best direct match because the page explicitly names that trip length and includes useful organization and security feature..."
3,beis_2023vs2026.txt,one_product_current,BEIS,01_2023_2024_original_vs_2026_current,4,"For packing organization, I would recommend BÉIS because it has the most detailed organization setup in the provided pages. Its interior includes a butterfly opening, easy-to-c...","For a straightforward 3-5 day carry-on, Delsey is the best direct match because the page explicitly names that trip length and includes useful organization and security feature..."
4,beis_2023vs2026.txt,one_product_current,BEIS,01_2023_2024_original_vs_2026_current,5,"For the best overall purchase confidence, I would choose Travelpro if the shopper is mainly flying on major domestic airlines. Its page provides exact overall dimensions, case ...","For a straightforward 3-5 day carry-on, Delsey is the best direct match because the page explicitly names that trip length and includes useful organization and security feature..."
5,delsey_2023vs2026.txt,one_product_current,Delsey,01_2023_2024_original_vs_2026_current,1,"For a 3-5 day trip, I would recommend the Travelpro Platinum Elite Carry-On Expandable Hardside Spinner as the most well-rounded pick: the page explicitly lists it for a 3-5 da...","Choose Travelpro for the best all-around feature set, durability, and mobility; choose BEIS for the richest packing organization and largest listed capacity; choose Monos for t..."
6,delsey_2023vs2026.txt,one_product_current,Delsey,01_2023_2024_original_vs_2026_current,2,Travelpro seems best for durability and protecting belongings based on the provided product-page text: its 100% polycarbonate shell is described as ultra-strong and able to fle...,"Choose Travelpro for the best all-around feature set, durability, and mobility; choose BEIS for the richest packing organization and largest listed capacity; choose Monos for t..."
7,delsey_2023vs2026.txt,one_product_current,Delsey,01_2023_2024_original_vs_2026_current,3,"Travelpro seems easiest to move overall because its PrecisionGlide System combines eight MagnaTrac self-aligning spinner wheels, a four-stop adjustable PowerScope Lite handle, ...","Choose Travelpro for the best all-around feature set, durability, and mobility; choose BEIS for the richest packing organization and largest listed capacity; choose Monos for t..."
8,delsey_2023vs2026.txt,one_product_current,Delsey,01_2023_2024_original_vs_2026_current,4,BEIS seems best for packing organization because its page lists a butterfly opening; a U-zip flap with a zip pocket and frosted PVC zip pocket; a detachable compression flap wi...,"Choose T

Focal current products detected by file:


,source_file,focal_current_product
0,beis_2023vs2026.txt,BEIS
5,delsey_2023vs2026.txt,Delsey
10,inusa_2023vs2026.txt,INUSA
15,monos_2023vs2026.txt,Monos
20,travel_2023vs2026.txt,Travelpro


## Load product-page texts

The PAIS-style score uses product-page text similarity to cited answer sentences.

If the text files are not found, the notebook still computes citation/rank/feature metrics, while the TF-IDF and n-gram parts of PAIS are set to zero.


In [18]:
# =============================
# Product text loading
# =============================

PRODUCT_FILE_STEMS = {
    "BEIS": "beis_carry_on_roller_filtered.txt",
    "Delsey": "delsey_chatelet_air_carry_on_plus_filtered.txt",
    "INUSA": "inusa_ally_carry_on_filtered.txt",
    "Monos": "monos_carry_on_filtered.txt",
    "Travelpro": "travelpro_platinum_elite_carry_on_filtered.txt",
}

def read_text_if_exists(path):
    if path is not None and path.exists():
        return path.read_text(encoding="utf-8", errors="replace")
    return ""

product_texts = {}
load_rows = []

for version_status, folder in [
    ("original_2023_2024", OLD_TEXT_DIR),
    ("current_2026", CURRENT_TEXT_DIR),
]:
    for product, filename in PRODUCT_FILE_STEMS.items():
        path = folder / filename if folder is not None else None
        text = read_text_if_exists(path)
        product_texts[(version_status, product)] = text
        load_rows.append({
            "version_status": version_status,
            "product": product,
            "path": str(path) if path is not None else "",
            "loaded": bool(text.strip()),
            "characters": len(text),
        })

product_text_load_status = pd.DataFrame(load_rows)
display(product_text_load_status)


,version_status,product,path,loaded,characters
0,original_2023_2024,BEIS,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\old\beis_carry_on_roller_filtered.txt,True,2375
1,original_2023_2024,Delsey,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\old\delsey_chatelet_air_carry_on_plus_filtered.txt,True,3572
2,original_2023_2024,INUSA,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\old\inusa_ally_carry_on_filtered.txt,True,1572
3,original_2023_2024,Monos,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\old\monos_carry_on_filtered.txt,True,1898
4,original_2023_2024,Travelpro,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\old\travelpro_platinum_elite_carry_on_filtered.txt,True,6179
5,current_2026,BEIS,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\current\beis_carry_on_roller_filtered.txt,True,2312
6,current_2026,Delsey,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\current\delsey_chatelet_air_carry_on_plus_filtered.txt,True,2102
7,current_2026,INUSA,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\current\inusa_ally_carry_on_filtered.txt,True,2128
8,current_2026,Monos,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\current\monos_carry_on_filtered.txt,True,3797
9,current_2026,Travelpro,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\data\filtered\current\travelpro_platinum_elite_carry_on_filtered.txt,True,5621


## Citation, rank, feature, and PAIS-style metrics


In [19]:
# =============================
# Citation and feature extraction
# =============================

PRODUCT_PATTERN = re.compile(r"\[(Monos|BÉIS|BEIS|Travelpro|Delsey|INUSA)\]", flags=re.IGNORECASE)

CANONICAL = {
    "monos": "Monos",
    "beis": "BEIS",
    "béis": "BEIS",
    "travelpro": "Travelpro",
    "delsey": "Delsey",
    "inusa": "INUSA",
}

FEATURE_PATTERNS = {
    "trip_length": r"\b(3-5|3–5|4-6|4–6|day trip|trip length|outfits?)\b",
    "overhead_bin_or_airline_fit": r"\b(overhead|carry-on|carry on|sizer|airline|domestic airlines?|flight|bin)\b",
    "dimensions": r"\b(dimensions?|inches|inch|in\.|cm|height|width|depth|linear)\b",
    "weight": r"\b(weight|lbs?|kg|lightweight|lighter|lift)\b",
    "capacity_volume": r"\b(capacity|volume|liters?|litres?|L\b|packing space|room)\b",
    "material_shell": r"\b(polycarbonate|PC/ABS|shell|hard shell|hardside|hard-sided|micro diamond|recycled materials?)\b",
    "durability_protection": r"\b(durable|durability|protect|protection|impact|cracking|corner|scuff|scratch|resistant|resilience|shock)\b",
    "wheels_mobility": r"\b(wheels?|spinner|rolling|roll|glide|gliding|maneuver|mobility|PrecisionGlide|MagnaTrac|Hinomoto|360)\b",
    "handle": r"\b(handle|telescopic|trolley|PowerScope|Contour Grip|grip|cushioned|soft-grip|GEL)\b",
    "lock_security": r"\b(TSA|lock|security|secure|Travel Sentry|combination)\b",
    "expansion": r"\b(expand|expandable|expansion|2-inch|2 inch|extra space)\b",
    "packing_organization": r"\b(compartment|compression|pocket|divider|strap|laundry|shoe|mesh|organize|organization|clamshell|butterfly|valuables|pouch)\b",
    "warranty_trial_return": r"\b(warranty|trial|return|returns|lifetime|10-year|100-day|promise|coverage)\b",
    "price_value": r"\b(price|priced|cost|lower-priced|value|sale|discount|investment)\b",
    "tradeoff_or_limitation": r"\b(limitation|limit|trade-off|tradeoff|may not|does not|caution|check|vary|not provide|less detail|however|but)\b",
}

def canonical_product(label: str) -> str:
    return CANONICAL.get(label.lower(), label)

def citation_sequence(text: str):
    return [canonical_product(m.group(1)) for m in PRODUCT_PATTERN.finditer(text)]

def split_sentences(text: str):
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]

def cited_sentences_for_product(answer: str, product: str):
    sentences = []
    for sent in split_sentences(answer):
        if product in set(citation_sequence(sent)):
            sentences.append(sent)
    return sentences

def categories_in_text(text: str):
    found = []
    for cat, pat in FEATURE_PATTERNS.items():
        if re.search(pat, text, flags=re.IGNORECASE):
            found.append(cat)
    return sorted(set(found))

def product_feature_categories(answer: str, product: str):
    cats = set()
    for sent in cited_sentences_for_product(answer, product):
        cats.update(categories_in_text(sent))
    return sorted(cats)

def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text or ""))

def simple_ngram_overlap(a: str, b: str, n: int = 2) -> float:
    def toks(x):
        return re.findall(r"\b[a-zA-Z0-9]+\b", (x or "").lower())
    ta, tb = toks(a), toks(b)
    if len(ta) < n or len(tb) < n:
        return 0.0
    nga = set(tuple(ta[i:i+n]) for i in range(len(ta)-n+1))
    ngb = set(tuple(tb[i:i+n]) for i in range(len(tb)-n+1))
    if not nga or not ngb:
        return 0.0
    return len(nga & ngb) / len(ngb)

def tfidf_similarity_to_product_text(product_text: str, cited_answer_text: str) -> float:
    if not SKLEARN_AVAILABLE:
        return 0.0
    if not product_text.strip() or not cited_answer_text.strip():
        return 0.0
    try:
        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
        X = vec.fit_transform([product_text, cited_answer_text])
        return float(cosine_similarity(X[0:1], X[1:2])[0, 0])
    except Exception:
        return 0.0

def build_question_product_metrics(parsed_df: pd.DataFrame, mode: str) -> pd.DataFrame:
    rows = []

    for _, row in parsed_df.iterrows():
        source_file = row["source_file"]
        run_type = row["run_type"]
        focal_current_product = row["focal_current_product"]
        qid = int(row["question_id"])
        answer = row["answer_text"]
        seq = citation_sequence(answer)

        unique_order = []
        for p in seq:
            if p not in unique_order:
                unique_order.append(p)

        first_unique = unique_order[0] if unique_order else None

        for product in PRODUCTS:
            if mode == "baseline":
                version_status = "original_2023_2024"
            else:
                version_status = "current_2026" if product == focal_current_product else "original_2023_2024"

            ref_count = seq.count(product)
            cited = ref_count > 0
            first_idx = seq.index(product) + 1 if cited else np.nan
            first_score = 1 / first_idx if cited else 0.0

            recommendation_rank_proxy = unique_order.index(product) + 1 if product in unique_order else N_PRODUCTS + 1
            rank_score_proxy = max((N_PRODUCTS + 1 - recommendation_rank_proxy) / N_PRODUCTS, 0)

            cited_sents = cited_sentences_for_product(answer, product)
            cited_answer_text = " ".join(cited_sents)
            product_text = product_texts.get((version_status, product), "")

            feature_cats = product_feature_categories(answer, product)
            feature_coverage = len(feature_cats) / len(FEATURE_PATTERNS)
            tfidf_sim = tfidf_similarity_to_product_text(product_text, cited_answer_text) if cited else 0.0
            ngram_overlap = simple_ngram_overlap(product_text, cited_answer_text, n=2) if cited else 0.0

            ref_component = min(ref_count / 3, 1)
            first_component = first_score

            # Product Answer Influence Score, adapted for this benchmark.
            # This is an observational proxy, not a causal or attention-based metric.
            pais = (
                0.25 * ref_component
                + 0.20 * first_component
                + 0.20 * feature_coverage
                + 0.20 * tfidf_sim
                + 0.15 * ngram_overlap
            )

            rows.append({
                "comparison": COMPARISON_NAME,
                "mode": mode,
                "source_file": source_file,
                "run_type": run_type,
                "focal_current_product": focal_current_product,
                "version_status_for_product": version_status,
                "question_id": qid,
                "product": product,
                "cited": int(cited),
                "ref_count": ref_count,
                "first_citation_index": first_idx,
                "first_citation_score": first_score,
                "recommendation_rank_proxy": recommendation_rank_proxy,
                "rank_score_proxy": rank_score_proxy,
                "top1_cited_flag": int(product == first_unique),
                "feature_categories": ";".join(feature_cats),
                "feature_count": len(feature_cats),
                "feature_coverage": feature_coverage,
                "tfidf_product_answer_similarity": tfidf_sim,
                "bigram_overlap_product_answer": ngram_overlap,
                "pais_product_answer_influence_score": pais,
                "answer_word_count": word_count(answer),
                "citation_sequence": " > ".join(seq),
                "cited_answer_sentences": cited_answer_text,
            })

    return pd.DataFrame(rows)

baseline_qp_metrics = build_question_product_metrics(baseline_answers, mode="baseline")
current_qp_metrics = build_question_product_metrics(current_answers, mode="one_product_current")

display(baseline_qp_metrics.head(15))
display(current_qp_metrics.head(15))


,comparison,mode,source_file,run_type,focal_current_product,version_status_for_product,question_id,product,cited,ref_count,...,top1_cited_flag,feature_categories,feature_count,feature_coverage,tfidf_product_answer_similarity,bigram_overlap_product_answer,pais_product_answer_influence_score,answer_word_count,citation_sequence,cited_answer_sentences
0,01_2023_2024_original_vs_2026_current,baseline,baseline.txt,baseline_all_original,None,original_2023_2024,1,Monos,0,0,...,0,,0,0.000000,0.000000,0.000000,0.000000,152,Delsey > Delsey > Travelpro,
1,01_2023_2024_original_vs_2026_current,baseline,baseline.txt,baseline_all_original,None,original_2023_2024,1,BEIS,0,0,...,0,,0,0.000000,0.000000,0.000000,0.000000,152,Delsey > Delsey > Travelpro,
2,01_2023_2024_original_vs_2026_current,baseline,baseline.txt,baseline_all_original,None,original_2023_2024,1,Travelpro,1,1,...,0,,0,0.000000,0.050716,0.000000,0.160143,152,Delsey > Delsey > Travelpro,[Travelpro]
3,01_2023_2024_original_vs_2026_current,baseline,baseline.txt,baseline_all_original,None,original_2023_2024,1,Delsey,1,2,...,1,capacity_volume;dimensions;expansion;material_shell;overhead_bin_or_airline_fit;packing_organization;tradeoff_or_limitation;trip_length;wheels_mobility,9,0.600000,0.124547,0.104651,0.527274,152,Delsey > Delsey > Travelpro,"[Delsey]\nIt is also practical for this trip length because it has dual compartments, compression cross straps, a zippered divider, mesh pockets, and included laundry and shoe ..."
4,01_2023_2024_original_vs_2026_current,baseline,baseline.txt,baseline_all_original,None,original_2023_2024,1,INUSA,0,0,...,0,,0,0.000000,0.000000,0.000000,0.000000,152,Delsey > Delsey > Travelpro,
5,01_2023_2024_original_vs_2026_current,baseline,baseline.txt,baseline_all_original,None,original_2023_2024,2,Monos,1,1,...,0,,0,0.000000,0.064614,0.000000,0.146256,132,Travelpro > Travelpro > Delsey > Monos,[Monos]
6,01_2023_2024_original_vs_2026_current,baseline,baseline.txt,baseline_all_original,None,original_2023_2024,2,BEIS,0,0,...,0,,0,0.000000,0.000000,0.000000,0.000000,132,Travelpro > Travelpro > Delsey > Monos,
7,01_2023_2024_original_vs_2026_current,baseline,baseline.txt,baseline_all_original,None,original_2023_2024,2,Travelpro,1,2,...,1,durability_protection;lock_security,2,0.133333,0.098992,0.230769,0.447747,132,Travelpro > Travelpro > Delsey > Monos,"[Travelpro]\nIt also adds aluminum corner armor for high-impact areas, a textured finish to reduce the visibility of scuffs or scratches, and a built-in TSA-approved lock. [Tra..."
8,01_2023_2024_original_vs_2026_current,baseline,baseline.txt,baseline_all_original,None,original_2023_2024,2,Delsey,1,1,...,0,durability_protection;lock_security;material_shell;tradeoff_or_limitation,4,0.266667,0.044573,0.035714,0.217605,132,Travelpro > Travelpro > Delsey > Monos,"[Delsey]\nMonos also looks protective, but the provided page gives fewer concrete protection details beyond an unbreakable polycarbonate shell, aerospace-grade polycarbonate, a..."
9,01_2023_2024_original_vs_2026_current,baseline,baseline.txt,baseline_all_original,None,original_2023_2024,2,INUSA,0,0,...,0,,0,0.000000,0.000000,0.000000,0.000000,132,Travelpro > Travelpro > Delsey > Monos,


,comparison,mode,source_file,run_type,focal_current_product,version_status_for_product,question_id,product,cited,ref_count,...,top1_cited_flag,feature_categories,feature_count,feature_coverage,tfidf_product_answer_similarity,bigram_overlap_product_answer,pais_product_answer_influence_score,answer_word_count,citation_sequence,cited_answer_sentences
0,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026.txt,one_product_current,BEIS,original_2023_2024,1,Monos,0,0,...,0,,0,0.000000,0.000000,0.000000,0.000000,182,Delsey > Delsey > Travelpro > BEIS,
1,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026.txt,one_product_current,BEIS,current_2026,1,BEIS,1,1,...,0,,0,0.000000,0.000000,0.000000,0.133333,182,Delsey > Delsey > Travelpro > BEIS,[BEIS]
2,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026.txt,one_product_current,BEIS,original_2023_2024,1,Travelpro,1,1,...,0,capacity_volume;dimensions;expansion;overhead_bin_or_airline_fit;tradeoff_or_limitation,5,0.333333,0.042411,0.026316,0.229096,182,Delsey > Delsey > Travelpro > BEIS,"[Travelpro] BÉIS is appealing for overpackers because it offers 49-61 L capacity and 2 inches of expandable space, but its page tells shoppers to check airline guidelines, so I..."
3,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026.txt,one_product_current,BEIS,original_2023_2024,1,Delsey,1,2,...,1,capacity_volume;dimensions;expansion;overhead_bin_or_airline_fit;packing_organization;tradeoff_or_limitation,6,0.400000,0.105548,0.130435,0.487341,182,Delsey > Delsey > Travelpro > BEIS,"[Delsey] It also has 44 L of volume, dual compartments with compression cross straps, a zippered divider, mesh pockets, and included laundry and shoe bags, so it gives enough s..."
4,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026.txt,one_product_current,BEIS,original_2023_2024,1,INUSA,0,0,...,0,,0,0.000000,0.000000,0.000000,0.000000,182,Delsey > Delsey > Travelpro > BEIS,
5,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026.txt,one_product_current,BEIS,original_2023_2024,2,Monos,1,1,...,0,,0,0.000000,0.064614,0.000000,0.136256,183,Travelpro > Travelpro > Delsey > Delsey > Monos,[Monos]
6,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026.txt,one_product_current,BEIS,current_2026,2,BEIS,0,0,...,0,,0,0.000000,0.000000,0.000000,0.000000,183,Travelpro > Travelpro > Delsey > Delsey > Monos,
7,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026.txt,one_product_current,BEIS,original_2023_2024,2,Travelpro,1,2,...,1,durability_protection;lock_security;warranty_trial_return;wheels_mobility,4,0.266667,0.067313,0.196429,0.462927,183,Travelpro > Travelpro > Delsey > Delsey > Monos,"[Travelpro] It also has a built-in TSA-approved lock, and its limited warranty covers functional defects involving wheels, zippers, extension handles, and carrying handles. [Tr..."
8,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026.txt,one_product_current,BEIS,original_2023_2024,2,Delsey,1,2,...,0,durability_protection;lock_security;material_shell;tradeoff_or_limitation;warranty_trial_return,5,0.333333,0.113190,0.196721,0.352146,183,Travelpro > Travelpro > Delsey > Delsey > Monos,[Delsey] Delsey also has a 100% polycarbonate shell and shock-absorbing corner protectors that safeguard against wear and tear. [Delsey] Monos is also durable on paper because ...
9,01_2023_2024_original_vs_2026_current,one_product_current,beis_2023vs2026.txt,one_product_current,BEIS,original_2023_2024,2,INUSA,0,0,...,0,,0,0.000000,0.000000,0.000000,0.000000,183,Travelpro > Travelpro > Delsey > Delsey > Monos,


## Product summaries

Baseline summary uses the direct all-original baseline file.

Current summary uses only the focal product rows from each one-product-current file.  
For example, BEIS current metrics are taken from `beis_2023vs2026.txt`.


In [20]:
# =============================
# Summaries
# =============================

def summarize_metrics(df: pd.DataFrame, label: str) -> pd.DataFrame:
    summary = (
        df.groupby(["product"])
        .agg(
            n_question_contexts=("question_id", "count"),
            n_source_files=("source_file", "nunique"),
            citation_rate=("cited", "mean"),
            mean_ref_count=("ref_count", "mean"),
            total_ref_count=("ref_count", "sum"),
            mean_first_citation_score=("first_citation_score", "mean"),
            mean_recommendation_rank_proxy=("recommendation_rank_proxy", "mean"),
            mean_rank_score_proxy=("rank_score_proxy", "mean"),
            top1_cited_share=("top1_cited_flag", "mean"),
            mean_feature_count=("feature_count", "mean"),
            mean_feature_coverage=("feature_coverage", "mean"),
            mean_tfidf_product_answer_similarity=("tfidf_product_answer_similarity", "mean"),
            mean_bigram_overlap_product_answer=("bigram_overlap_product_answer", "mean"),
            mean_pais_product_answer_influence_score=("pais_product_answer_influence_score", "mean"),
        )
        .reset_index()
    )
    summary.insert(0, "metric_source", label)
    return summary

baseline_product_summary = summarize_metrics(baseline_qp_metrics, "baseline_all_original")

# Keep only focal-current rows for the current summary.
current_focal_rows = current_qp_metrics[
    current_qp_metrics["product"] == current_qp_metrics["focal_current_product"]
].copy()

current_product_summary = summarize_metrics(current_focal_rows, "one_product_current_focal")

display(baseline_product_summary)
display(current_product_summary)


,metric_source,product,n_question_contexts,n_source_files,citation_rate,mean_ref_count,total_ref_count,mean_first_citation_score,mean_recommendation_rank_proxy,mean_rank_score_proxy,top1_cited_share,mean_feature_count,mean_feature_coverage,mean_tfidf_product_answer_similarity,mean_bigram_overlap_product_answer,mean_pais_product_answer_influence_score
0,baseline_all_original,BEIS,5,1,0.4,0.6,3,0.266667,4.2,0.36,0.2,0.4,0.026667,0.012310,0.024339,0.114779
1,baseline_all_original,Delsey,5,1,0.8,1.0,5,0.383333,2.8,0.64,0.2,2.8,0.186667,0.058215,0.036406,0.214437
2,baseline_all_original,INUSA,5,1,0.2,0.2,1,0.033333,5.4,0.12,0.0,0.0,0.000000,0.012327,0.000000,0.025799
3,baseline_all_original,Monos,5,1,0.6,0.8,4,0.150000,4.0,0.40,0.0,1.4,0.093333,0.037434,0.008602,0.124110
4,baseline_all_original,Travelpro,5,1,0.8,1.6,8,0.666667,2.2,0.76,0.6,2.0,0.133333,0.064153,0.137607,0.326805


,metric_source,product,n_question_contexts,n_source_files,citation_rate,mean_ref_count,total_ref_count,mean_first_citation_score,mean_recommendation_rank_proxy,mean_rank_score_proxy,top1_cited_share,mean_feature_count,mean_feature_coverage,mean_tfidf_product_answer_similarity,mean_bigram_overlap_product_answer,mean_pais_product_answer_influence_score
0,one_product_current_focal,BEIS,5,1,0.8,1.0,5,0.333333,3.6,0.48,0.2,0.8,0.053333,0.024581,0.052422,0.173446
1,one_product_current_focal,Delsey,5,1,0.8,0.8,4,0.250000,3.2,0.56,0.0,2.8,0.186667,0.031869,0.010534,0.161954
2,one_product_current_focal,INUSA,5,1,0.2,0.2,1,0.050000,5.6,0.08,0.0,0.0,0.000000,0.002007,0.000000,0.027068
3,one_product_current_focal,Monos,5,1,0.8,0.8,4,0.366667,3.0,0.60,0.2,0.8,0.053333,0.037587,0.027368,0.162289
4,one_product_current_focal,Travelpro,5,1,0.8,1.2,6,0.666667,2.4,0.72,0.6,2.0,0.133333,0.062950,0.121236,0.290775


## Current-vs-baseline delta by product

This is the main output.

Positive delta means the product became more visible or more prominent when its own page was replaced by the 2026 current version, compared with the pure all-original baseline.


In [21]:
# =============================
# Current - baseline delta
# =============================

def make_current_vs_baseline_delta(baseline_summary: pd.DataFrame, current_summary: pd.DataFrame) -> pd.DataFrame:
    b = baseline_summary.set_index("product")
    c = current_summary.set_index("product")

    products = sorted(set(b.index).intersection(set(c.index)))
    rows = []

    for product in products:
        base = b.loc[product]
        cur = c.loc[product]

        rows.append({
            "comparison": COMPARISON_NAME,
            "product": product,
            "baseline_n_question_contexts": int(base["n_question_contexts"]),
            "current_n_question_contexts": int(cur["n_question_contexts"]),

            "baseline_citation_rate": base["citation_rate"],
            "current_citation_rate": cur["citation_rate"],
            "delta_citation_rate": cur["citation_rate"] - base["citation_rate"],

            "baseline_mean_ref_count": base["mean_ref_count"],
            "current_mean_ref_count": cur["mean_ref_count"],
            "delta_mean_ref_count": cur["mean_ref_count"] - base["mean_ref_count"],

            "baseline_first_citation_score": base["mean_first_citation_score"],
            "current_first_citation_score": cur["mean_first_citation_score"],
            "delta_first_citation_score": cur["mean_first_citation_score"] - base["mean_first_citation_score"],

            "baseline_mean_recommendation_rank_proxy": base["mean_recommendation_rank_proxy"],
            "current_mean_recommendation_rank_proxy": cur["mean_recommendation_rank_proxy"],
            "rank_improvement_current_minus_baseline": base["mean_recommendation_rank_proxy"] - cur["mean_recommendation_rank_proxy"],

            "baseline_rank_score_proxy": base["mean_rank_score_proxy"],
            "current_rank_score_proxy": cur["mean_rank_score_proxy"],
            "delta_rank_score_proxy": cur["mean_rank_score_proxy"] - base["mean_rank_score_proxy"],

            "baseline_top1_cited_share": base["top1_cited_share"],
            "current_top1_cited_share": cur["top1_cited_share"],
            "delta_top1_cited_share": cur["top1_cited_share"] - base["top1_cited_share"],

            "baseline_feature_coverage": base["mean_feature_coverage"],
            "current_feature_coverage": cur["mean_feature_coverage"],
            "delta_feature_coverage": cur["mean_feature_coverage"] - base["mean_feature_coverage"],

            "baseline_pais": base["mean_pais_product_answer_influence_score"],
            "current_pais": cur["mean_pais_product_answer_influence_score"],
            "delta_pais": cur["mean_pais_product_answer_influence_score"] - base["mean_pais_product_answer_influence_score"],
        })

    delta = pd.DataFrame(rows)

    # Composite descriptive score.
    # This is not a causal estimate; it summarizes observed answer-behavior movement.
    delta["current_page_advantage_score"] = (
        0.25 * delta["delta_rank_score_proxy"]
        + 0.20 * delta["delta_citation_rate"]
        + 0.20 * delta["delta_first_citation_score"]
        + 0.15 * delta["delta_top1_cited_share"]
        + 0.10 * delta["delta_feature_coverage"]
        + 0.10 * delta["delta_pais"]
    )

    delta["direction_label"] = np.select(
        [
            delta["current_page_advantage_score"] > 0.05,
            delta["current_page_advantage_score"] < -0.05,
        ],
        [
            "current_2026_stronger",
            "baseline_2023_2024_stronger",
        ],
        default="similar_or_small_change",
    )

    return delta.sort_values("current_page_advantage_score", ascending=False)

current_vs_baseline_delta_by_product = make_current_vs_baseline_delta(
    baseline_product_summary,
    current_product_summary,
)

display(current_vs_baseline_delta_by_product)


,comparison,product,baseline_n_question_contexts,current_n_question_contexts,baseline_citation_rate,current_citation_rate,delta_citation_rate,baseline_mean_ref_count,current_mean_ref_count,delta_mean_ref_count,...,current_top1_cited_share,delta_top1_cited_share,baseline_feature_coverage,current_feature_coverage,delta_feature_coverage,baseline_pais,current_pais,delta_pais,current_page_advantage_score,direction_label
3,01_2023_2024_original_vs_2026_current,Monos,5,5,0.6,0.8,0.2,0.8,0.8,0.0,...,0.2,0.2,0.093333,0.053333,-4.000000e-02,0.124110,0.162289,0.038179,0.163151,current_2026_stronger
0,01_2023_2024_original_vs_2026_current,BEIS,5,5,0.4,0.8,0.4,0.6,1.0,0.4,...,0.2,0.0,0.026667,0.053333,2.666667e-02,0.114779,0.173446,0.058667,0.131867,current_2026_stronger
2,01_2023_2024_original_vs_2026_current,INUSA,5,5,0.2,0.2,0.0,0.2,0.2,0.0,...,0.0,0.0,0.000000,0.000000,0.000000e+00,0.025799,0.027068,0.001269,-0.006540,similar_or_small_change
4,01_2023_2024_original_vs_2026_current,Travelpro,5,5,0.8,0.8,0.0,1.6,1.2,-0.4,...,0.6,0.0,0.133333,0.133333,2.775558e-17,0.326805,0.290775,-0.036030,-0.013603,similar_or_small_change
1,01_2023_2024_original_vs_2026_current,Delsey,5,5,0.8,0.8,0.0,1.0,0.8,-0.2,...,0.0,-0.2,0.186667,0.186667,0.000000e+00,0.214437,0.161954,-0.052483,-0.081915,baseline_2023_2024_stronger


## Overall summary


In [22]:
overall_current_vs_baseline_summary = pd.DataFrame([{
    "comparison": COMPARISON_NAME,
    "purpose": "Check how much the actual product page changed over time",
    "interpretation": "General website/text change",
    "baseline_file": str(BASELINE_FILE),
    "comparison_dir": str(COMPARISON_DIR),
    "n_products": len(current_vs_baseline_delta_by_product),
    "mean_delta_citation_rate": current_vs_baseline_delta_by_product["delta_citation_rate"].mean(),
    "mean_delta_rank_score_proxy": current_vs_baseline_delta_by_product["delta_rank_score_proxy"].mean(),
    "mean_rank_improvement_current_minus_baseline": current_vs_baseline_delta_by_product["rank_improvement_current_minus_baseline"].mean(),
    "mean_delta_first_citation_score": current_vs_baseline_delta_by_product["delta_first_citation_score"].mean(),
    "mean_delta_top1_cited_share": current_vs_baseline_delta_by_product["delta_top1_cited_share"].mean(),
    "mean_delta_feature_coverage": current_vs_baseline_delta_by_product["delta_feature_coverage"].mean(),
    "mean_delta_pais": current_vs_baseline_delta_by_product["delta_pais"].mean(),
    "mean_current_page_advantage_score": current_vs_baseline_delta_by_product["current_page_advantage_score"].mean(),
    "n_current_2026_stronger": int((current_vs_baseline_delta_by_product["direction_label"] == "current_2026_stronger").sum()),
    "n_baseline_2023_2024_stronger": int((current_vs_baseline_delta_by_product["direction_label"] == "baseline_2023_2024_stronger").sum()),
    "n_similar_or_small_change": int((current_vs_baseline_delta_by_product["direction_label"] == "similar_or_small_change").sum()),
}])

display(overall_current_vs_baseline_summary)


,comparison,purpose,interpretation,baseline_file,comparison_dir,n_products,mean_delta_citation_rate,mean_delta_rank_score_proxy,mean_rank_improvement_current_minus_baseline,mean_delta_first_citation_score,mean_delta_top1_cited_share,mean_delta_feature_coverage,mean_delta_pais,mean_current_page_advantage_score,n_current_2026_stronger,n_baseline_2023_2024_stronger,n_similar_or_small_change
0,01_2023_2024_original_vs_2026_current,Check how much the actual product page changed over time,General website/text change,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\baseline\baseline.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\shopping_answers\01_2023_2024_original_vs_2026_current,5,0.12,0.032,0.16,0.033333,0.0,-0.002667,0.00192,0.038592,2,1,2


## Save CSV outputs


In [23]:
baseline_answers.to_csv(OUTPUT_DIR / "baseline_parsed_answers.csv", index=False, encoding="utf-8-sig")
current_answers.to_csv(OUTPUT_DIR / "current_parsed_answers_by_file.csv", index=False, encoding="utf-8-sig")
baseline_qp_metrics.to_csv(OUTPUT_DIR / "baseline_question_product_metrics.csv", index=False, encoding="utf-8-sig")
current_qp_metrics.to_csv(OUTPUT_DIR / "current_question_product_metrics.csv", index=False, encoding="utf-8-sig")
baseline_product_summary.to_csv(OUTPUT_DIR / "baseline_product_summary.csv", index=False, encoding="utf-8-sig")
current_product_summary.to_csv(OUTPUT_DIR / "current_product_summary.csv", index=False, encoding="utf-8-sig")
current_vs_baseline_delta_by_product.to_csv(OUTPUT_DIR / "current_vs_baseline_delta_by_product.csv", index=False, encoding="utf-8-sig")
overall_current_vs_baseline_summary.to_csv(OUTPUT_DIR / "overall_current_vs_baseline_summary.csv", index=False, encoding="utf-8-sig")
product_text_load_status.to_csv(OUTPUT_DIR / "product_text_load_status.csv", index=False, encoding="utf-8-sig")

print("Saved CSV files to:")
print(OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", p.name)


Saved CSV files to:
C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\results\tables\01_2023_2024_original_vs_2026_current_baseline_based
- baseline_parsed_answers.csv
- baseline_product_summary.csv
- baseline_question_product_metrics.csv
- current_parsed_answers_by_file.csv
- current_product_summary.csv
- current_question_product_metrics.csv
- current_vs_baseline_delta_by_product.csv
- overall_current_vs_baseline_summary.csv
- product_text_load_status.csv


## Quick interpretation table

Use this table first when writing the result.


In [24]:
cols = [
    "product",
    "direction_label",
    "current_page_advantage_score",
    "delta_rank_score_proxy",
    "delta_citation_rate",
    "delta_first_citation_score",
    "delta_top1_cited_share",
    "delta_feature_coverage",
    "delta_pais",
]
display(current_vs_baseline_delta_by_product[cols])


,product,direction_label,current_page_advantage_score,delta_rank_score_proxy,delta_citation_rate,delta_first_citation_score,delta_top1_cited_share,delta_feature_coverage,delta_pais
3,Monos,current_2026_stronger,0.163151,0.20,0.2,0.216667,0.2,-4.000000e-02,0.038179
0,BEIS,current_2026_stronger,0.131867,0.12,0.4,0.066667,0.0,2.666667e-02,0.058667
2,INUSA,similar_or_small_change,-0.006540,-0.04,0.0,0.016667,0.0,0.000000e+00,0.001269
4,Travelpro,similar_or_small_change,-0.013603,-0.04,0.0,0.000000,0.0,2.775558e-17,-0.036030
1,Delsey,baseline_2023_2024_stronger,-0.081915,-0.08,0.0,-0.133333,-0.2,0.000000e+00,-0.052483
